# Fig 3: KV-Transfer Overhead vs Input Length

**Output:** `model_outputs/<MODEL_SHORT>/paper/section3/fig3/network_overhead.{pdf,png}`

Paths are resolved via `MODEL_DATA_DIR` and `GPU_MON_ROOT` from `profiling/config.py`.

### Call order
1. **Collect** — checks `GPU_MON_ROOT/<MODEL_SHORT>/pd_disagg_300W/<L>/` for each L; runs `profiling/launch_disagg_profile.sh` for any missing normal-L values. Power cap must be 300W (`sudo nvidia-smi -i 0,1 -pl 300`). To collect long-L values (10240–65536) uncomment `LONG_L_VALUES` in the collect cell and set `INCLUDE_LONG=1`.
2. **Plot** — `scripts/plot_network_overhead.py`

In [1]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)

import sys; sys.path.insert(0, str(REPO_ROOT / "config"))
from config import MODEL, GPU_TYPE, MODEL_DATA_DIR, GPU_MON_ROOT, MODEL_SHORT

print(f"GPU  : {GPU_TYPE}")
print(f"Model: {MODEL}")


def run(cmd):
    buf = []
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return "".join(buf)

GPU  : A40
Model: Qwen/Qwen3.6-27B


In [2]:
NORMAL_L_VALUES = [128, 256, 512, 1024, 2048, 4096, 6144, 8192]
LONG_L_VALUES   = [10240, 12288, 16384, 20480, 24576, 28672, 32768,
                   40960, 49152, 57344, 65536]

#? Add LONG_L_VALUES to run full sweep
L_VALUES = NORMAL_L_VALUES
# L_VALUES = NORMAL_L_VALUES + LONG_L_VALUES


disagg_base = GPU_MON_ROOT / MODEL_SHORT / "pd_disagg_300W"

def _missing(Ls):
    return [
        L for L in Ls
        if not (disagg_base / str(L) / "dcgmi_trace.tsv").exists()
        or not (disagg_base / str(L) / "decoder_forward_start_time.csv").exists()
    ]

missing_normal = _missing(NORMAL_L_VALUES)
missing_long   = _missing(LONG_L_VALUES)

if missing_normal:
    print(f"[{MODEL_SHORT}] missing normal L values: {missing_normal} — launching profiling sweep")
    run(["bash", str(REPO_ROOT / "profiling/launch_disagg_profile.sh")])
else:
    print(f"[{MODEL_SHORT}] all {len(NORMAL_L_VALUES)} normal L values complete")

if missing_long:
    print(f"[{MODEL_SHORT}] missing long L values: {missing_long} — run with INCLUDE_LONG=1 to collect")
else:
    print(f"[{MODEL_SHORT}] all {len(LONG_L_VALUES)} long L values complete")

[Qwen3.6-27B] missing normal L values: [128, 256, 512, 1024, 2048, 4096, 6144, 8192] — launching profiling sweep
Model:       Qwen/Qwen3.6-27B
Output base: /data/projects/eicchen/conserve_project/conserve/profiling/gpu_profiling/A40/Qwen3.6-27B/pd_disagg_300W
Normal L:    128 256 512 1024 2048 4096 6144 8192
Long L:      10240 12288 16384 20480 24576 28672 32768 40960 49152 57344 65536
Running L:   128 256 512 1024 2048 4096 6144 8192  (INCLUDE_LONG=0)

--- L=128: starting ---
MODEL:          Qwen/Qwen3.6-27B
IN_TOKEN_SIZE:  128
LOG_DIR:        /data/projects/eicchen/conserve_project/conserve/profiling/gpu_profiling/A40/Qwen3.6-27B/pd_disagg_300W/128
Waiting for server on port 7200...
Using model: Qwen/Qwen3.6-27B
decoder device: CUDA_VISIBLE_DEVICES=1
Using model: Qwen/Qwen3.6-27B
prefiller device: CUDA_VISIBLE_DEVICES=0
INFO:     Started server process [3014296]
INFO:     Waiting for application startup.
Added decoder client: http://localhost:7200, localhost-[7300]-[7400]
[2026-06-29

KeyboardInterrupt: 

In [3]:
%matplotlib inline
%run ../scripts/plot_network_overhead.py

L=   128: n= 2032  p50=  4.22  p99=  5.93  net%= 18.6
L=   256: n= 2032  p50=  4.49  p99=  6.00  net%= 19.5
L=   512: n= 2032  p50=  4.85  p99=  6.93  net%= 19.4
L=1024: skipped ([Errno 2] No such file or directory: '/data/projects/eicchen/conserve_project/conserve/profiling/gpu_profiling/A40/Qwen3-0.6B/pd_disagg_300W/1024/decoder_forward_start_time.csv')


ValueError: need at least one array to concatenate

In [ ]:
from IPython.display import Image

Image(str(MODEL_DATA_DIR / "paper" / "section3" / "fig3" / "network_overhead.png"))